# Автоматическая семантическая разметка многозначной лексики с использованием RuBERT

In [1]:
!pip install -q transformers sentencepiece razdel conllu tqdm


In [2]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel


In [3]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
rubert = AutoModel.from_pretrained(MODEL_NAME)

rubert = rubert.to(device)
rubert.eval()

print(device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cpu


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

## Загрузка данных

In [4]:
!pip install conllu


In [5]:
import re
SEM_PATTERN = re.compile(r"sem=\[([^\]]+)\]")

def extract_sem_tags(misc): # вытаскиваем семантические теги
    if misc is None:
        return []

    # случай: conllu вернул dict
    if isinstance(misc, dict):
        sem_val = misc.get("sem")
        if not sem_val:
            return []
        sem_val = sem_val.strip("[]")
        return [t.strip() for t in sem_val.split(",")]

    # случай: строка
    match = SEM_PATTERN.search(str(misc))
    if not match:
        return []

    return [t.strip() for t in match.group(1).split(",")]


In [6]:
pos_map = {
    "NOUN":  "S",       # существительное
    "ADJ":   "A",       # прилагательное
    "VERB":  "V",       # глагол
    "ADV":   "ADV",     # наречие
    "PRON":  "SPRO",    # местоимение
    "ADVPRO": "ADV",    # местоименное наречие
    "NUM":   "NUM",     # числительное
    "PROPN": "S",       # имя собственное → существительное

    "ADP":   "PR",      # предлог
    "CCONJ": "CONJ",    # сочинительный союз
    "SCONJ": "CONJ",    # подчинительный союз
    "PART":  "PART",    # частица
    "INTJ":  "INTJ",    # междометие

    "DET":   "APRO",    # определительное/местоименное прилагательное
    "AUX":   "V",       # вспомогательный глагол
    "SYM":   "SYM",     # символ
    "X":     "X",       # другое / неизвестное
    }

def conllu_to_w2v_format(lemma, upos): # POS MyStem → POS UD
    pos = pos_map.get(upos)

    if pos is None:
        return lemma

    return f"{lemma}_{pos}"

# TODO: посмотреть отдельные слова, например, здесь ADVPRO не переводится в ADV
# [w for w in wv.key_to_index if w.startswith("лингвистика_")][:20]


In [7]:
from conllu import parse

def parse_conllu(text): # берем предложение/текст и делаем словарь из словарей с признаками слова
    sentences = parse(text)
    parsed = []

    for sent in sentences:
        tokens = []
        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            UD_lemma = conllu_to_w2v_format(token["lemma"], token["upos"])

            tokens.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "UD_lemma": UD_lemma,
                "UD_pos": pos_map.get(token["upos"]),
                "feats": token["feats"],
                "head": token["head"],
                "deprel": token["deprel"],
                "sem": sem_tags
            })

        parsed.append(tokens)

    return parsed

# выдает набор токенов в таком формате:
# {'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}


In [8]:
from google.colab import drive
drive.mount('/content/drive')

conllu_file = "/content/drive/MyDrive/Colab Notebooks/Аспирантура/conllu/handlabeled/GramEval2020-test-fiction.conllu"

with open(conllu_file, "r", encoding="utf-8") as f:
    conllu_data = f.read()
    


Mounted at /content/drive


In [9]:
conllu_text = parse_conllu(conllu_data)
# print(conllu_text)

for token in conllu_text[0]:
  print(token)


{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}
{'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}
{'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}
{'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}
{'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'VerbFo

In [10]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def text_to_vector(text, model=rubert, tokenizer=tokenizer):
    if not isinstance(text, str) or not text.strip():
        return None

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    vector = mean_pooling(
        outputs.last_hidden_state,
        encoding["attention_mask"]
    )[0]

    return vector.cpu().numpy().astype(np.float32)


In [11]:
def extract_examples(conllu_text, target_tags=None):
    sentences = parse(conllu_text)
    results = []

    if target_tags:
        target_tags = [t.strip().lower() for t in target_tags]

    for sent in sentences:
        sentence_text = sent.metadata.get("text", "")

        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            sem_tags = [t.lower() for t in sem_tags]

            if not sem_tags:
                continue

            if target_tags:
                if not any(tag in sem_tags for tag in target_tags):
                    continue

            results.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "sem": sem_tags,
                "sentence": sentence_text,
                "token_id": token["id"]
            })

    return results


In [12]:
target_tags = ["tool"]

searching_for_tags = extract_examples(conllu_data, target_tags)
for found_word_with_tag in searching_for_tags:
  print(found_word_with_tag)


{'form': 'автомат', 'lemma': 'автомат', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 8}
{'form': 'столик', 'lemma': 'столик', 'upos': 'NOUN', 'sem': ['concr', 'furn', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 10}
{'form': 'экран', 'lemma': 'экран', 'upos': 'NOUN', 'sem': ['concr', 'device', 'part', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 20}
{'form': 'телевизора', 'lemma': 'телевизор', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 21}
{'form': 'дверь', 'lemma': 'дверь', 'upos': 'NOUN', 'sem': ['concr', 'constr', 'part',

In [13]:
from datasets import load_dataset

dictionary_definitions = load_dataset("snagbreac/russian-reverse-dictionary-train-data") # датасет со словарными определениями


README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/294007 [00:00<?, ? examples/s]

In [14]:
print(dictionary_definitions['train'][1000])
# формат словаря: {'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


{'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


## Векторизация текста

## Контекст target

In [15]:
def get_context_window(sentence_tokens, target_index, window_size=3):
    start = max(0, target_index - window_size)
    end = min(len(sentence_tokens), target_index + window_size + 1)
    return sentence_tokens[start:end]


In [16]:
def get_token_char_spans(tokens):
    spans = []
    position = 0

    for token in tokens:
        form = str(token.get("form", ""))
        start = position
        end = start + len(form)
        spans.append((start, end))
        position = end + 1

    return spans


In [17]:
def target_to_vector(sentence_tokens, target_index, model=rubert, tokenizer=tokenizer):
    context_tokens = get_context_window(
        sentence_tokens,
        target_index,
        window_size=3
    )

    target_local_index = target_index - max(0, target_index - 3)

    text = " ".join(
        str(token.get("form", ""))
        for token in context_tokens
    )

    spans = get_token_char_spans(context_tokens)

    target_start, target_end = spans[target_local_index]

    encoding = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=128
    )

    offset_mapping = encoding.pop("offset_mapping")[0].tolist()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    hidden_states = outputs.last_hidden_state[0]

    target_indices = [
        i for i, (start, end) in enumerate(offset_mapping)
        if start < target_end and end > target_start
    ]

    if not target_indices:
        return None

    vector = hidden_states[target_indices].mean(dim=0)

    return vector.cpu().numpy().astype(np.float32)


## Косинусная близость

In [18]:
def cosine_sim(vector1, vector2):
    if vector1 is None or vector2 is None:
        return None

    vector1 = np.asarray(vector1)
    vector2 = np.asarray(vector2)

    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return float(np.dot(vector1, vector2) / (norm1 * norm2))


## Векторизация словарных дефиниций

In [19]:
def build_word2vectorized_def(dataset, model=rubert, tokenizer=tokenizer):
    word2vectorized_def = {}
    definition_vector_cache = {}

    definition_id = 0
    total = len(dataset)

    for i, item in enumerate(dataset, start=1):
        word = str(item["word"]).lower()
        definition = item["definition"]
        dictionary_name = item.get("df")

        if definition in definition_vector_cache:
            definition_vector = definition_vector_cache[definition]
        else:
            definition_vector = text_to_vector(
                definition,
                model,
                tokenizer
            )
            definition_vector_cache[definition] = definition_vector

        definition_id += 1

        definition_data = {
            "id": definition_id,
            "dict": dictionary_name,
            "definition": definition,
            "vector": definition_vector
        }

        word2vectorized_def.setdefault(word, []).append(definition_data)

        if i % 1000 == 0:
            print(f"Обработано: {i} / {total}")

    return word2vectorized_def


In [20]:
word2vectorized_def = build_word2vectorized_def(dictionary_definitions["train"])

with open("word2vectorized_def_rubert.pkl", "wb") as f:
    pickle.dump(word2vectorized_def, f)


Обработано: 1000 / 294007
Обработано: 2000 / 294007
Обработано: 3000 / 294007
Обработано: 4000 / 294007
Обработано: 5000 / 294007
Обработано: 6000 / 294007
Обработано: 7000 / 294007
Обработано: 8000 / 294007
Обработано: 9000 / 294007
Обработано: 10000 / 294007
Обработано: 11000 / 294007
Обработано: 12000 / 294007
Обработано: 13000 / 294007
Обработано: 14000 / 294007
Обработано: 15000 / 294007
Обработано: 16000 / 294007
Обработано: 17000 / 294007
Обработано: 18000 / 294007


: 

In [1]:
# Загрузка:
with open("word2vectorized_def_rubert.pkl", "rb") as f:
    word2vectorized_def = pickle.load(f)


FileNotFoundError: [Errno 2] No such file or directory: 'word2vectorized_def_rubert.pkl'

## Семантические теги

In [ ]:
def descriptions_to_tag_vectors(tag_descriptions, model=rubert, tokenizer=tokenizer):
    tag2vector = {}

    for tag, description in tag_descriptions.items():
        vector = text_to_vector(
            description,
            model,
            tokenizer
        )

        tag2vector[tag] = {
            "description": description,
            "vector": vector
        }

    return tag2vector


In [ ]:
def predict_tags(definition_vector, tag2vector, threshold=0.5):
    if definition_vector is None:
        return [], {}

    tag_similarities = {}

    for tag, data in tag2vector.items():
        tag_vector = data.get("vector")

        if tag_vector is None:
            continue

        similarity = cosine_sim(
            definition_vector,
            tag_vector
        )

        tag_similarities[tag] = similarity

    predicted_tags = [
        tag
        for tag, similarity in tag_similarities.items()
        if similarity >= threshold
    ]

    predicted_tags.sort(
        key=lambda tag: tag_similarities[tag],
        reverse=True
    )

    return predicted_tags, tag_similarities


## Выбор словарной дефиниции

In [ ]:
def find_best_definition(context_vector, list_of_definitions):
    if context_vector is None or not list_of_definitions:
        return []

    ranked_definitions = []

    for definition_data in list_of_definitions:
        definition_vector = definition_data.get("vector")

        if definition_vector is None:
            continue

        similarity = cosine_sim(
            context_vector,
            definition_vector
        )

        ranked_definitions.append({
            "id": definition_data.get("id"),
            "dict": definition_data.get("dict"),
            "definition": definition_data["definition"],
            "vector": definition_vector,
            "similarity": similarity
        })

    ranked_definitions.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return ranked_definitions


## Обработка предложения

In [ ]:
def is_significant_token(token):
    upos = token.get("upos")

    if upos in {"PUNCT", "SYM"}:
        return False

    form = token.get("form", "")

    return bool(form and re.search(r"\w", form, flags=re.UNICODE))


In [2]:
def process_sentence(
    sentence_tokens,
    context,
    model,
    tokenizer,
    word2vectorized_def,
    tag2vector,
    threshold=0.5,
    window_size=3
):
    results = []

    significant_indices = [
        i for i, token in enumerate(sentence_tokens)
        if is_significant_token(token)
    ]

    for target_index in significant_indices:

        token = sentence_tokens[target_index]

        form = token.get("form")
        lemma = token.get("lemma")
        gold_tags = token.get("sem", [])

        local_indices = significant_indices[
            max(0, significant_indices.index(target_index) - window_size):
            significant_indices.index(target_index) + window_size + 1
        ]

        context_tokens = [
            sentence_tokens[i]
            for i in local_indices
        ]

        target_position = local_indices.index(target_index)

        context_vector = target_to_vector(
            context_tokens,
            target_position,
            model=model,
            tokenizer=tokenizer
        )

        local_context = " ".join(
            t.get("form", "")
            for t in context_tokens
        )

        base_result = {
            "form": form,
            "lemma": lemma,
            "upos": token.get("upos"),
            "context": context,
            "context_window": [
                t.get("form")
                for t in context_tokens
            ],
            "local_context": local_context,
            "gold_tags": gold_tags,
            "definition": None,
            "definition_similarity": None,
            "predicted_tags": [],
            "tag_similarities": {},
            "status": None
        }

        if context_vector is None:
            base_result["status"] = "no_context_vector"
            results.append(base_result)
            continue

        list_of_definitions = word2vectorized_def.get(
            str(lemma).lower(),
            []
        )

        if not list_of_definitions:
            base_result["status"] = "no_definition"
            results.append(base_result)
            continue

        ranked_definitions = find_best_definition(
            context_vector,
            list_of_definitions
        )

        if not ranked_definitions:
            base_result["status"] = "no_definition_vector"
            results.append(base_result)
            continue

        best_definition = ranked_definitions[0]

        predicted_tags, tag_similarities = predict_tags(
            best_definition["vector"],
            tag2vector,
            threshold=threshold
        )

        base_result.update({
            "definition": best_definition["definition"],
            "definition_similarity": best_definition["similarity"],
            "predicted_tags": predicted_tags,
            "tag_similarities": tag_similarities,
            "status": "classified"
        })

        results.append(base_result)

    return results


## Обработка корпуса

In [3]:
def process_corpus(
    conllu_text,
    model,
    tokenizer,
    word2vectorized_def,
    tag2vector,
    threshold=0.5,
    limit=None
):
    all_results = []

    data = conllu_text if limit is None else conllu_text[:limit]

    for sentence_tokens in tqdm(data):
        context = " ".join(
            token["form"]
            for token in sentence_tokens
        )

        sentence_results = process_sentence(
            sentence_tokens=sentence_tokens,
            context=context,
            model=model,
            tokenizer=tokenizer,
            word2vectorized_def=word2vectorized_def,
            tag2vector=tag2vector,
            threshold=threshold
        )

        all_results.extend(sentence_results)

    return all_results


## Запуск

In [ ]:
# tag2vector = descriptions_to_tag_vectors(
#     tag_descriptions,
#     rubert,
#     tokenizer
# )

# results = process_corpus(
#     conllu_text=conllu_text,
#     model=rubert,
#     tokenizer=tokenizer,
#     word2vectorized_def=word2vectorized_def,
#     tag2vector=tag2vector,
#     threshold=0.5
# )


## Сохранение результатов

In [ ]:
# with open("results_rubert.pkl", "wb") as f:
#     pickle.dump(results, f)

# df_results = pd.DataFrame(results)
# df_results.to_pickle("results_rubert_dataframe.pkl")
# df_results.head()


## Метрики

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, classification_report


def calculate_metrics(results):
    valid_results = [
        r for r in results
        if r.get("status") == "classified"
    ]

    if not valid_results:
        return None

    y_true = [r.get("gold_tags", []) for r in valid_results]
    y_pred = [r.get("predicted_tags", []) for r in valid_results]

    mlb = MultiLabelBinarizer()
    y_true_bin = mlb.fit_transform(y_true)
    y_pred_bin = mlb.transform(y_pred)

    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true_bin,
        y_pred_bin,
        average="micro",
        zero_division=0
    )

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true_bin,
        y_pred_bin,
        average="macro",
        zero_division=0
    )

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1
    }


# metrics = calculate_metrics(results)
# metrics


In [ ]:
def calculate_class_metrics(results):
    valid_results = [
        r for r in results
        if r.get("status") == "classified"
    ]

    if not valid_results:
        return pd.DataFrame()

    y_true = [r.get("gold_tags", []) for r in valid_results]
    y_pred = [r.get("predicted_tags", []) for r in valid_results]

    mlb = MultiLabelBinarizer()
    y_true_bin = mlb.fit_transform(y_true)
    y_pred_bin = mlb.transform(y_pred)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_bin,
        y_pred_bin,
        average=None,
        zero_division=0
    )

    return pd.DataFrame({
        "tag": mlb.classes_,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support
    }).sort_values("support", ascending=False)


# class_metrics = calculate_class_metrics(results)
# class_metrics
